In [126]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf

In [127]:
# Indlæs data
GDP = pd.read_csv("GDP.csv", skiprows=4)
ODR = pd.read_csv("age-dependency-ratio-old.csv")
TFP = pd.read_csv("total-factor-productivity.csv")
HC = pd.read_excel("pwt110.xlsx", sheet_name="Data")
PWT = HC[['country','year','hc','ctfp','rgdpe','pop']]

In [128]:
# Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land,
# hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

# Udvælger relevante variabler fra Penn World Table
PWT = HC[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Beregner BNP per capita ud fra rgdpe og befolkning
PWT['gdp_pc'] = PWT['rgdpe'] / PWT['pop']

# Udtrækker observationsåret 2002 og beholder de variable,
# der skal bruges som initiale forklarende variable i tværsnittet
base2002 = PWT[PWT['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()

# Omdøber variablerne, så det tydeligt fremgår, at de er målt i 2002
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# Udtrækker TFP i slutåret 2020
tfp2020 = PWT[PWT['year'] == 2020][['country','ctfp']].copy()

# Omdøber TFP-variablen for 2020
tfp2020.columns = ['country','TFP2020']

# Sammenfletter 2002- og 2020-data, så hvert land får både initial og slut-TFP
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')

# Konstruerer long-difference outcome som ændringen i log(TFP) fra 2002 til 2020
Tvaersnit['GrowthTFP'] = np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])

# Udtrækker old dependency ratio i startåret 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()

# Omdøber variablerne, så de matcher merge-strukturen
odr2002.columns = ['country', 'ODR2002']

# Sammenfletter ODR med tværsnittet, så hvert land får sin initiale ODR-værdi
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')

# Fjerner observationer med manglende værdier
Tvaersnit = Tvaersnit.dropna()

In [129]:
print(Tvaersnit.describe())

           HC2002     TFP2002     GDPpc2002     TFP2020   GrowthTFP  \
count  108.000000  108.000000    108.000000  108.000000  108.000000   
mean     2.500757    0.674189  18667.498333    0.637384   -0.014267   
std      0.659964    0.296350  17809.178627    0.220858    0.304781   
min      1.088122    0.167584    728.561351    0.200132   -1.073842   
25%      2.082968    0.403153   4843.455089    0.471157   -0.224303   
50%      2.576247    0.652114  11166.490440    0.625006   -0.045691   
75%      2.967072    0.883181  32387.904671    0.819311    0.183162   
max      3.598119    1.394810  84592.225755    1.236640    0.942964   

          ODR2002  
count  108.000000  
mean    12.640748  
std      7.701428  
min      1.583193  
25%      6.062487  
50%      8.882906  
75%     20.099692  
max     28.059470  


### Model 1: Growth_TFP = α + β ODR2002 + ε

Har lande med højere ODR i 2002 lavere TFP-vækst 2002–2020?

In [133]:
model1 = smf.ols("GrowthTFP ~ ODR2002", data=Tvaersnit).fit()
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.060
Method:                 Least Squares   F-statistic:                     7.814
Date:                Mon, 20 Apr 2026   Prob (F-statistic):            0.00616
Time:                        14:12:52   Log-Likelihood:                -20.581
No. Observations:                 108   AIC:                             45.16
Df Residuals:                     106   BIC:                             50.53
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1168      0.055      2.130      0.0

Negativ og statistisk signifikant sammenhæng mellem ODR i 2002 og TFP-vækst 2002–2020.

ODR2002 = -0.0104 
- Hvis et land havde 1 point højere old dependency ratio i 2002, var TFP-væksten over perioden i gennemsnit ca. 0.010 lavere.

Signifikans
p = 0.006
- Signifikant på 1%-niveau - Så dette er ikke bare tilfældig støj.

R² = 0.069

- Forklarer ca. 7% af variationen.

Konklussion 

- Vores baseline-estimat viser, at lande med højere initial old dependency ratio oplevede lavere vækst i total factor productivity fra 2002 til 2020.
- Baseline regressionen viser en negativ sammenhæng mellem initial aldring og efterfølgende produktivitetsvækst. Koefficienten på ODR er -0.0104 og statistisk signifikant (p = 0.006), hvilket indikerer, at lande med ældre befolkninger voksede langsommere i TFP over perioden.

### Model 2: Growt_hTFP = α + βODR2002 + γ HC2002 + ε

Påvirker aldring stadig TFP-vækst, når vi tager højde for human capital?

In [134]:
model2 = smf.ols("GrowthTFP ~ ODR2002 + HC2002", data=Tvaersnit).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.074
Model:                            OLS   Adj. R-squared:                  0.056
Method:                 Least Squares   F-statistic:                     4.191
Date:                Mon, 20 Apr 2026   Prob (F-statistic):             0.0177
Time:                        14:16:52   Log-Likelihood:                -20.274
No. Observations:                 108   AIC:                             46.55
Df Residuals:                     105   BIC:                             54.60
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0321      0.123      0.262      0.7

ODR2002 = -0.0135

før

- -0.0104

Nu:

- -0.0135

Så når vi holder humankapital konstant, bliver aldringseffekten endnu tydeligere.

Et land med 1 point højere ODR i 2002 havde ca. 0.0135 lavere TFP-vækst 2002–2020, alt andet lige.

p = 0.016

- signifikant på 5%-niveau.

HC2002

- Koefficient = 0.0498

- p = 0.441

Ikke signifikant.

- Vi finder ikke selvstændig evidens for, at initial humankapital forklarer TFP-vækst i denne simple model.

Efter kontrol for human capital forbliver koefficienten på ODR negativ og statistisk signifikant. Dette indikerer, at sammenhængen mellem aldring og lavere produktivitetsvækst ikke alene kan forklares af forskelle i humankapital.

### Model 3: GrowthTFP = α + β ODR200 2 γ HC2002 + δ log(GDPpc2002) + ε

Påvirker aldring stadig produktivitetsvækst, når vi sammenligner lande med samme humankapital og samme udviklingsniveau ved startåret?

Er aldring stadig vigtig, når to lande starter lige rige?

In [135]:
model3 = smf.ols("GrowthTFP ~ ODR2002 + HC2002 + np.log(GDPpc2002)", data=Tvaersnit).fit()
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.177
Model:                            OLS   Adj. R-squared:                  0.153
Method:                 Least Squares   F-statistic:                     7.433
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           0.000147
Time:                        14:23:57   Log-Likelihood:                -13.931
No. Observations:                 108   AIC:                             35.86
Df Residuals:                     104   BIC:                             46.59
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.8431      0.25

ODR2002
- Koefficient = -0.0086

p = 0.114

- Ikke signifikant ved 5%.

Når vi tager højde for udviklingsniveau og human capital: aldringseffekten svækkes statistisk

HC2002
- Koefficient = +0.1819

p = 0.012

Lande med højere human capital i 2002 oplevede højere TFP-vækst.

log(GDPpc2002)
- Koefficient = -0.1298

p < 0.001

Rige lande voksede langsommere end fattige lande.


En del af ODR-effekten hænger sammen med, at ældre lande ofte også er rige og modne økonomier.